# Exploration notebook
### This notebook is for initial pass of the overall dataset and data quality check
- Check null values and duplicated enrollment_id
- Check invalid course_id
- Check invalid date formats
- Check numeric value columns 

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime
import numpy as np

In [ ]:
# Explore the existing db schema
conn = sqlite3.connect("../data/courses.db")

tables = conn.execute(
    "select name, sql from sqlite_master where type='table'"
).fetchall()

for name, sql in tables:
    print(f"\nTable: {name}")
    print(sql)

conn.close()


Table: courses
CREATE TABLE courses (
        course_id INTEGER PRIMARY KEY,
        name TEXT,
        description TEXT,
        prerequisites TEXT
    )

Table: enrollments
CREATE TABLE enrollments (
        enrollment_id INTEGER PRIMARY KEY,
        participant_id TEXT,
        participant_name TEXT,
        course_id INTEGER,
        course_date DATE,
        amount REAL,
        subsidy REAL,
        credits_used REAL
    )


In [ ]:
# check tables
courses = pd.read_sql_query(
    "select * from courses limit 10",
    sqlite3.connect("../data/courses.db")
)
display(courses)

,course_id,name,description,prerequisites
0,1,CS100 Introduction to Computer Science,Foundational course covering computational thi...,NaN
1,2,CS101 Data structures,"Study of arrays, linked lists, trees, graphs, ...",NaN
2,3,CS200 Databases,"Covers relational database design, SQL, normal...","[1,2]"
3,4,CS201 Networking,"Explores network protocols, TCP/IP stack, rout...",[3]
4,5,CS300 Systems,"Advanced course on operating systems, concurre...",[4]


In [ ]:
# check tables
enrollments = pd.read_sql_query(
    "select * from enrollments limit 10",
    sqlite3.connect("../data/courses.db")
)
display(enrollments)

,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used


In [8]:
csv_path = Path("../data/enrolments.csv")

df = pd.read_csv(csv_path)

print(df.shape)
print(df.dtypes)
display(df.head(10))

(34, 8)
enrollment_id         int64
participant_id          str
participant_name        str
course_id             int64
course_date             str
amount              float64
subsidy             float64
credits_used        float64
dtype: object


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used
0,1,P001,Alice Tan,1,15/01/2024,500.0,200.0,300.0
1,2,P002,Bob Lee,1,2024-01-15,500.0,250.0,250.0
2,3,P007,Grace Koh,1,20 Jan 2024,500.0,250.0,250.0
3,4,P009,Irene Chua,1,2024-01-25,500.0,200.0,300.0
4,5,P003,Charlie Ng,2,01-02-2024,600.0,300.0,300.0
5,6,P001,Alice Tan,2,2024/02/01,600.0,200.0,400.0
6,7,P004,Diana Lim,1,10 Feb 2024,500.0,150.0,350.0
7,8,P007,Grace Koh,2,2024-02-15,600.0,300.0,300.0
8,9,P009,Irene Chua,2,20/02/2024,600.0,250.0,350.0
9,10,P002,Bob Lee,3,2024-03-05,750.0,400.0,350.0


In [ ]:
# check if there is null value and duplicate enrollment_id
print(df.isna().sum())
print("Duplicate enrollment IDs:", df["enrollment_id"].duplicated().sum())

enrollment_id       0
participant_id      0
participant_name    0
course_id           0
course_date         0
amount              0
subsidy             0
credits_used        0
dtype: int64
Duplicate enrollment IDs: 0


In [ ]:
# check course date format
print(df["course_date"].unique())

# the date format is quite a mess, need to focus to clean up

<StringArray>
[ '15/01/2024',  '2024-01-15', '20 Jan 2024',  '2024-01-25',  '01-02-2024',
  '2024/02/01', '10 Feb 2024',  '2024-02-15',  '20/02/2024',  '2024-03-05',
  '15/03/2024',  '2024-03-20', '25 Mar 2024',  '2024/04/01',  '01-04-2024',
  '2024-04-05',  '10/04/2024',  '2024-04-15',  '22/04/2024', '01 May 2024',
  '2023-02-29',  '2024/05/15',  '20-05-2024',  '25/05/2024', '01 Jun 2024',
  '1900-06-10',  '2024-06-15',  '2024/06/20',  '01-07-2024',  '2024-07-01',
 '05 Jul 2024',  '10/07/2024',  '2024-07-15',  '2024-07-20']
Length: 34, dtype: str


In [ ]:
# as there is no foreign key constraint of course_id in courses table, so need to check invalid course_id in data
valid_course_ids = pd.read_sql_query(
    "select course_id from courses",
    sqlite3.connect("../data/courses.db")
)["course_id"]

invalid_courses = df[
    ~df["course_id"].isin(valid_course_ids)
]

display(invalid_courses)

,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used


In [ ]:
# convert different date format to YYYY-MM-DD
# return none for invalid date format
DATE_FORMATS = [
    "%d/%m/%Y",  # 15/01/2024
    "%Y-%m-%d",  # 2024-01-15
    "%d %b %Y",  # 20 Jan 2024
    "%d-%m-%Y",  # 01-02-2024
    "%Y/%m/%d",  # 2024/02/01
]

def parse_course_date(value):
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(str(value).strip(), fmt).date().isoformat()
        except ValueError:
            continue

    return None

In [15]:
df["parsed_course_date"] = df["course_date"].apply(parse_course_date)

In [ ]:
# display invalid date
# 2023 is not leap year so no 02-29
invalid_dates = df[df["parsed_course_date"].isna()]

display(invalid_dates[["enrollment_id", "course_date"]])

,enrollment_id,course_date
20,21,2023-02-29


In [ ]:
# find valid but suspicious year like 1900
df["course_year"] = pd.to_datetime(
    df["parsed_course_date"]
).dt.year

suspicious_dates = df[
    (df["course_year"] < 2020) |
    (df["course_year"] > 2030)
]

display(suspicious_dates[
    ["enrollment_id", "course_date", "parsed_course_date"]
])

,enrollment_id,course_date,parsed_course_date
25,26,1900-06-10,1900-06-10


In [ ]:
# check numeric columns
financial_cols = ["amount", "subsidy", "credits_used"]

for col in financial_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# non-numeric, missing or infinite values
invalid_numeric = ~np.isfinite(
    df[financial_cols].to_numpy(dtype=float)
).all(axis=1)

# negative values
negative_values = (df[financial_cols] < 0).any(axis=1)

In [ ]:
# validation rule: total subsidy and credits cannot exceed amount
excess_coverage = (
    df["subsidy"] + df["credits_used"]
    > df["amount"] + 1e-9
)

print("Invalid numeric:", invalid_numeric.sum())
print("Negative values:", negative_values.sum())
print("Excess coverage:", excess_coverage.sum())

display(df[
    invalid_numeric | negative_values | excess_coverage
])

Invalid numeric: 0
Negative values: 0
Excess coverage: 0


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used,parsed_course_date,course_year
